In [ ]:
import torch
from torch import nn
import torchvision
from torch.utils import data
from torchvision import datasets, transforms
from torch.nn import functional as F
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import Subset
import random
import numpy as np
seed=42
batch_size=128
epoch=15
lr=5e-4
k=5
loss=nn.CrossEntropyLoss()
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # 让CUDA尽可能使用确定性算法
    # torch.backends.cudnn.deterministic = True
    # torch.backends.cudnn.benchmark = False


set_seed(seed)
def try_gpu(i=0):
    if torch.cuda.device_count()>=i+1:
        return torch.device(f'cuda:{i}')
    return torch.device('cpu')
device=try_gpu()

In [ ]:
def data_maker():
    mean = (0.485, 0.456, 0.406)
    std = (0.229, 0.224, 0.225)

    train_transform = transforms.Compose([
        # 填充成40×40，再随机裁剪回32×32
        transforms.RandomResizedCrop(
            224,
            scale=(0.7, 1.0)
        ),

        # 50%概率水平翻转
        transforms.RandomHorizontalFlip(
            p=0.5
        ),

        # 轻微改变颜色
        # transforms.ColorJitter(
        #     brightness=0.2,
        #     contrast=0.2,
        #     saturation=0.2,
        #     hue=0.05
        # ),

        transforms.ToTensor(),

        transforms.Normalize(
            mean=mean,
            std=std
        )
    ])

    test_transform = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),

        # 测试集必须使用同样的标准化参数
        transforms.Normalize(
            mean=mean,
            std=std
        )
    ])

    train_dataset = torchvision.datasets.CIFAR10(
        root="../cifar10-data",
        train=True,
        transform=train_transform,
        download=False
    )
    valid_dataset = torchvision.datasets.CIFAR10(
            root="../cifar10-data",
            train=True,
            transform=test_transform,
            download=False
        )

    test_dataset = torchvision.datasets.CIFAR10(
        root="../cifar10-data",
        train=False,
        transform=test_transform,
        download=False
    )

    return train_dataset,valid_dataset,test_dataset

In [ ]:
def data_loader(dataset,batch_size,flag):
    return data.DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=flag,
        num_workers=8,
        pin_memory=True,
        persistent_workers=True
    )


In [ ]:
def net_maker():
    net=torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.DEFAULT) #
    net.fc=nn.Linear(net.fc.in_features,10)
    return net
    

In [ ]:
train_dataset,valid_dataset,test_dataset=data_maker()
def get_k_fold_indices(k,train_dataset,valid_dataset=None,seed=42):
    if valid_dataset==None:
        valid_dataset=train_dataset
    label=np.array(train_dataset.targets)
    k_fold=StratifiedKFold(n_splits=k,shuffle=True,random_state=seed)
    fold_indices = list(
    k_fold.split(np.zeros(len(label)),label)
    )
    return fold_indices
fold_indices=get_k_fold_indices(k,train_dataset,valid_dataset)
def get_k_fold_data(fold_indices,i,train_dataset,valid_dataset):
    train_indices,valid_indices=fold_indices[i]
    train_data=Subset(train_dataset,train_indices.tolist())
    valid_data=Subset(valid_dataset,valid_indices.tolist())
    return train_data,valid_data
def init(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)
    
    


In [ ]:
for i in range(1):#先跑一折看看如何
    net=net_maker().to(device)
    train_data,valid_data=get_k_fold_data(fold_indices,i,train_dataset,valid_dataset)
    net.fc.apply(init)
    train_iter=data_loader(train_data,batch_size,True)
    valid_iter=data_loader(valid_data,batch_size,False)
    main_parameter=[]
    for name,param in net.named_parameters():
        if not name.startswith('fc.'):
            main_parameter.append(param)
    trainer=torch.optim.SGD([
        {
            'params':main_parameter,
            'lr':lr
        },
        {
            'params':net.fc.parameters(),
            'lr':lr*10
        }
    ],weight_decay=1e-4,momentum=0.9)
    scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(trainer,T_max=epoch,eta_min=1e-6)
    for j in range(epoch):
        net.train()
        train_loss=0.0
        train_count=0
        train_acc=0
        for x,y in train_iter:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            trainer.zero_grad()
            logit=net(x)
            l=loss(logit,y)
            train_loss+=l.item()*len(y)
            train_count+=len(y)
            train_acc+=(logit.argmax(dim=1)==y).sum().item()
            l.backward()
            trainer.step()
        scheduler.step()
        print(f"第{i+1}折的第{j+1}个epoch的训练loss：{train_loss/train_count:.5f},准确率:{train_acc/train_count:.5f}")
        net.eval()
        with torch.no_grad():
            valid_loss=0.0
            valid_count=0
            valid_acc=0.0
            for x,y in valid_iter:
                x = x.to(device, non_blocking=True)
                y = y.to(device, non_blocking=True)
                logit=net(x)
                l=loss(logit,y)
                valid_loss+=l.item()*len(y)
                valid_count+=len(y)
                valid_acc+=(logit.argmax(dim=1)==y).sum().item()
            print(f"第{i+1}折的第{j+1}个epoch的验证loss：{valid_loss/valid_count},准确率:{valid_acc/valid_count}")